In [4]:
!pip install "ag2[openai]" autogen

from google.colab import userdata
import os

# Securely fetch the key from Colab Secrets
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("Groq API key loaded successfully.")
except userdata.SecretNotFoundError:
    print("Error: Please add 'GROQ_API_KEY' to your Colab secrets.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 32.4 MB/s eta 0:00:00
Groq API key loaded successfully.


In [13]:
# Operating-system helpers for API keys
import os

# AutoGen building blocks for the hotel desk
from autogen import (
    AssistantAgent,
    UserProxyAgent,
    GroupChat,
    GroupChatManager,
    register_function,
)

def build_llm_config():
    """Build llm_config strictly from Groq env vars."""
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("GROQ_API_KEY not found. Please add it to your Colab secrets.")

    model = os.getenv("LLM_MODEL", "openai/gpt-oss-120b")
    config = {
        "model": model,
        "api_key": api_key,
        "base_url": "https://api.groq.com/openai/v1",
        "price": [0.0, 0.0]
    }
    return {"config_list": [config], "temperature": 0}

llm_config = build_llm_config()

HOTEL_STAYS = {
    "BK-7781": {"guest": "Rohit Sharma", "room": "412", "check_in": "2026-04-10", "check_out": "2026-04-13"},
    "BK-9002": {"guest": "Anita Desai", "room": "918", "check_in": "2026-04-11", "check_out": "2026-04-14"},
}

def lookup_guest_stay(booking_id: str) -> str:
    """Look up a hotel stay by booking id."""
    bid = (booking_id or "").strip().upper()
    row = HOTEL_STAYS.get(bid)
    if not row: return f"STAY_NOT_FOUND: {bid}"
    return f"STAY {bid}: guest={row['guest']}; room={row['room']}"

def create_complaint_ticket(category: str, summary: str, room: str) -> str:
    """Create a complaint ticket after successful stay lookup."""
    return f"TICKET_CREATED: HT-{category.upper()}-{room}"

intake = AssistantAgent(
    name="IntakeAgent",
    llm_config=llm_config,
    system_message="You are hotel intake. If booking id and room are present, say 'INTAKE_READY'. Otherwise, ask for missing details. Do not do anything else."
)

classifier = AssistantAgent(
    name="ClassifierAgent",
    llm_config=llm_config,
    system_message="Classify the complaint into one category: room_comfort, housekeeping, or billing. Reply ONLY with the category name."
)

clerk = AssistantAgent(
    name="DeskClerkAgent",
    llm_config=llm_config,
    system_message="You are the desk clerk. Use lookup_guest_stay first. If found, use create_complaint_ticket. After TICKET_CREATED, say 'Your ticket has been created' and end your message with 'TERMINATE'."
)

desk = UserProxyAgent(
    name="HotelDeskRunner",
    human_input_mode="NEVER",
    code_execution_config=False,
    is_termination_msg=lambda m: "TERMINATE" in (m.get("content") or ""),
    llm_config=False
)

register_function(lookup_guest_stay, caller=clerk, executor=desk, name="lookup_guest_stay", description="Lookup stay.")
register_function(create_complaint_ticket, caller=clerk, executor=desk, name="create_complaint_ticket", description="Create ticket.")

def hotel_speaker_select(last_speaker, groupchat):
    last = groupchat.messages[-1] if groupchat.messages else {}
    content = str(last.get("content", ""))

    if "TERMINATE" in content: return None
    if "tool_calls" in str(last).lower(): return desk
    if last_speaker is desk:
        if "STAY" in content or "TICKET" in content: return clerk
        return intake
    if last_speaker is intake:
        if "INTAKE_READY" in content: return classifier
        return intake
    if last_speaker is classifier: return clerk
    return clerk

groupchat = GroupChat(agents=[desk, intake, classifier, clerk], messages=[], max_round=12, speaker_selection_method=hotel_speaker_select)
manager = GroupChatManager(groupchat=groupchat, llm_config=llm_config, is_termination_msg=lambda m: "TERMINATE" in (m.get("content") or ""))

def run_hotel_desk(complaint: str):
    print(f"\nProcessing: {complaint}")
    desk.initiate_chat(manager, message=complaint, clear_history=True)

if __name__ == "__main__":
    run_hotel_desk("AC not working in room 412, booking BK-7781")


Processing: AC not working in room 412, booking BK-7781
HotelDeskRunner (to chat_manager):

AC not working in room 412, booking BK-7781

--------------------------------------------------------------------------------

Next speaker: IntakeAgent

IntakeAgent (to chat_manager):

INTAKE_READY

--------------------------------------------------------------------------------

Next speaker: ClassifierAgent

ClassifierAgent (to chat_manager):

room_comfort

--------------------------------------------------------------------------------

Next speaker: DeskClerkAgent

DeskClerkAgent (to chat_manager):

***** Suggested tool call (fc_4f08bc7a-3931-41b1-9d7d-2cd9c4fa564b): lookup_guest_stay *****
Arguments: 
{"booking_id":"BK-7781"}
********************************************************************************************

--------------------------------------------------------------------------------

Next speaker: HotelDeskRunner


>>>>>>>> EXECUTING FUNCTION lookup_guest_stay...
Call ID: f